# 52 · Data Analyst — SQL Question to Dashboard

**Persona:** Data analyst. **Tools exercised:** `SQLTool` (real SQLite), `DashboardRenderTool`.

Unlike the other persona notebooks, this one does **not** stub the database — we spin up an in-memory SQLite engine, seed it with a small sales table, then drive `SQLTool` against it for real. That gives you one notebook where the tool actually executes a query.

Workflow:

1. Create + seed an in-memory SQLite database.
2. Inspect the schema via the tool (list_tables + describe_table).
3. Run a SQL query through the tool to answer a question.
4. Render the answer as a dashboard HTML artifact.


## Setup

In [ ]:
from pathlib import Path

def _find_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        return cwd
    candidate = cwd / 'notebooks'
    return candidate if candidate.is_dir() else cwd
WORKSPACE = _find_notebooks_dir() / '_analyst_workspace'
WORKSPACE.mkdir(parents=True, exist_ok=True)
print('workspace:', WORKSPACE)


## 1 · Pick a model

In [ ]:
# from shipit_agent.llms import build_llm_from_settings
# llm = build_llm_from_settings({'provider': 'bedrock',
#     'model': 'bedrock/openai.gpt-oss-120b-1:0'}, provider='bedrock')
# llm = build_llm_from_settings({'provider': 'litellm',
#     'model': 'openai/gpt-4o-mini'}, provider='litellm')
# from shipit_agent.llms import LiteLLMProxyChatLLM
# llm = LiteLLMProxyChatLLM(model='gpt-4o-mini',
#     api_base='https://litellm.internal', api_key='sk-proxy')

from shipit_agent.llms import SimpleEchoLLM
llm = SimpleEchoLLM()
print('llm:', type(llm).__name__)


## 2 · Seed a real in-memory SQLite

This runs a small `orders` table straight through SQLAlchemy. `SQLTool` is lazy about SQLAlchemy — it only imports it on first `run()`. If you don't have SQLAlchemy installed (`pip install 'shipit-agent[sql]'`), the tool returns a structured `sqlalchemy_missing` output rather than raising.


In [ ]:
try:
    from sqlalchemy import create_engine, text
    HAS_SQLALCHEMY = True
except ImportError:  # pragma: no cover - ci installs it
    HAS_SQLALCHEMY = False
    print('SQLAlchemy not installed — skipping seed. '
          'Install with: pip install shipit-agent[sql]')

engine = None
if HAS_SQLALCHEMY:
    engine = create_engine('sqlite:///:memory:')
    with engine.begin() as conn:
        conn.execute(text('''
            CREATE TABLE orders (
              id INTEGER PRIMARY KEY,
              region TEXT NOT NULL,
              product TEXT NOT NULL,
              amount_cents INTEGER NOT NULL,
              placed_at TEXT NOT NULL
            )
        '''))
        seed = [
            (1, 'EMEA', 'Pro',   9900,  '2026-04-18'),
            (2, 'AMER', 'Pro',   9900,  '2026-04-19'),
            (3, 'EMEA', 'Team', 39900,  '2026-04-19'),
            (4, 'APAC', 'Pro',   9900,  '2026-04-20'),
            (5, 'AMER', 'Team', 39900,  '2026-04-20'),
            (6, 'EMEA', 'Pro',   9900,  '2026-04-21'),
            (7, 'AMER', 'Enterprise', 199900, '2026-04-21'),
            (8, 'APAC', 'Team', 39900,  '2026-04-22'),
            (9, 'EMEA', 'Enterprise', 199900, '2026-04-22'),
            (10,'AMER', 'Pro',   9900,  '2026-04-23'),
        ]
        for row in seed:
            conn.execute(text(
                'INSERT INTO orders (id, region, product, amount_cents, placed_at) '
                'VALUES (:id, :region, :product, :amount_cents, :placed_at)'
            ), dict(zip(['id', 'region', 'product', 'amount_cents', 'placed_at'], row)))
    print('seeded 10 orders into :memory: sqlite')


## 3 · Inspect schema via the tool

In [ ]:
from shipit_agent.tools.sql import SQLTool
from shipit_agent.tools.base import ToolContext

sql_tool = SQLTool(engine=engine)
ctx = ToolContext(prompt='analyst', state={})

tables = sql_tool.run(ctx, action='list_tables')
print('--- list_tables ---')
print(tables.text)

schema = sql_tool.run(ctx, action='describe_table', table='orders')
print()
print('--- describe_table orders ---')
print(schema.text)


## 4 · Run the analyst's question

"What is revenue by region this week, and which region leads?"


In [ ]:
answer = sql_tool.run(
    ctx, action='query',
    sql=('SELECT region, SUM(amount_cents) / 100 AS revenue_usd, '
         'COUNT(*) AS order_count '
         'FROM orders '
         "WHERE placed_at >= '2026-04-20' "
         'GROUP BY region ORDER BY revenue_usd DESC'),
)
print(answer.text)
rows = answer.metadata.get('rows', [])


## 5 · Render the answer as a dashboard

In [ ]:
from shipit_agent.tools.dashboard_render import DashboardRenderTool

dash = DashboardRenderTool(workspace_root=WORKSPACE)

palette = ['#185fa5', '#1d9e75', '#534ab7']
metric_items = [
    {'label': r['region'], 'value': f"${r['revenue_usd']:,}",
     'sub': f"{r['order_count']} orders"}
    for r in rows
]
max_rev = max((r['revenue_usd'] for r in rows), default=1) or 1
bars = [
    {'label': r['region'],
     'pct': int(round(r['revenue_usd'] / max_rev * 100)),
     'color': palette[i % len(palette)]}
    for i, r in enumerate(rows)
]

result = dash.run(
    ToolContext(prompt='analyst',
                 state={'artifact_workspace_root': str(WORKSPACE)}),
    title='Weekly Revenue by Region',
    subtitle='Week of 2026-04-20 — sourced from orders table',
    lang='en',
    sections=[
        {'type': 'metrics', 'title': 'Region snapshot',
         'columns': len(metric_items), 'items': metric_items},
        {'type': 'bars', 'title': 'Revenue share', 'items': bars},
        {'type': 'verdict', 'title': 'Answer',
         'text': (f'**{rows[0]["region"]}** leads the week with '
                  f'${rows[0]["revenue_usd"]:,} in revenue across '
                  f'{rows[0]["order_count"]} orders.') if rows else '(no data)'},
    ],
    export=True,
)
print(result.text)
print('artifact path:', result.metadata.get('path'))


## Next steps

* Swap the in-memory SQLite URL for your real warehouse — any SQLAlchemy URL works: `postgresql://...`, `bigquery://project`, `snowflake://...`.
* See `docs-app/content/source/tools/sql.md` for the safety posture (`allow_writes=False` by default, `max_rows` cap, timeout).
* Let a `GoalAgent` pick the SQL: give it the tool + `schema_summary` hint and ask it to answer the question in plain English.
